In [5]:
# ============================================================================
# 5-MIN RT FEATURES  ->  one row per (dt, hr, node_num)
#
#   spine    : spp_lmp.settlement_location_rt_hourly   (dt, hr, node_num, rt_value)
#   features : spp_lmp.sl_rt_5_min  aggregated over the 12 intervals of (dt-LAG, hr, node)
#
# Availability: bidding day D, the newest settled RT is D-2, so every 5-min feature is
# stamped onto the target dt it is legal to bid with (LAG=2), same hr, same node.
#
# The 5-min table is a ROLLING ~15-DAY WINDOW (463,392 rows/day). It cannot be backfilled,
# so run this daily and append to a permanent table -- see run_incremental().
# ============================================================================
import sys; sys.path.append('/var/www/python/Prod/nighthawk/')
import numpy as np, pandas as pd
from nighthawk.util import sql_functions as sf

LAG       = 2                 # days back: the newest RT you may bid with
VALUE_COL = 'rt_value'        # or 'congestional_value' / 'marginalloss_value'
PREFIX    = 'rt5m'            # feature name prefix; change with VALUE_COL
FIVE_MIN  = 'spp_lmp.sl_rt_5_min'
HOURLY    = 'spp_lmp.settlement_location_rt_hourly'


def five_min_coverage():
    """Which dts the rolling 5-min table currently holds (and so which targets are buildable)."""
    d = sf.download_df_from_sql_db(f'SELECT dt, COUNT(*) n FROM {FIVE_MIN} GROUP BY dt ORDER BY dt')
    d['dt'] = pd.to_datetime(d['dt']).dt.strftime('%Y-%m-%d')
    return d


def agg_5min_one_dt(src_dt, value_col=VALUE_COL, prefix=PREFIX, nodes=None):
    """The 12 intervals of each (src_dt, hr, node) collapsed to one row. Aggregation runs in
    SQL -- a day is 463k rows in, 38.6k rows out, ~1.5s."""
    node_clause = ''
    if nodes is not None:
        node_clause = f" AND node_num IN ({','.join(str(int(n)) for n in nodes)})"
    q = f"""
        SELECT dt AS src_dt, hr, node_num,
               COUNT(*)                 AS {prefix}_n,
               MAX({value_col})         AS {prefix}_max,
               MIN({value_col})         AS {prefix}_min,
               AVG({value_col})         AS {prefix}_mean,
               STDDEV_SAMP({value_col}) AS {prefix}_std,
               MAX(ABS({value_col}))    AS {prefix}_absmax
        FROM {FIVE_MIN}
        WHERE dt = '{src_dt}'{node_clause}
        GROUP BY dt, hr, node_num
    """
    a = sf.download_df_from_sql_db(q)
    if not len(a):
        return a
    a['src_dt'] = pd.to_datetime(a['src_dt']).dt.strftime('%Y-%m-%d')
    for c in (f'{prefix}_max', f'{prefix}_min', f'{prefix}_mean', f'{prefix}_std', f'{prefix}_absmax'):
        a[c] = pd.to_numeric(a[c], errors='coerce')
    # derived: what the hourly average hides
    a[f'{prefix}_range'] = a[f'{prefix}_max'] - a[f'{prefix}_min']     # intra-hour swing
    a[f'{prefix}_spike'] = a[f'{prefix}_max'] - a[f'{prefix}_mean']    # peak above the hourly mean
    a[f'{prefix}_drop']  = a[f'{prefix}_mean'] - a[f'{prefix}_min']    # trough below it
    return a


def hourly_one_dt(dt, nodes=None):
    node_clause = f" AND node_num IN ({','.join(str(int(n)) for n in nodes)})" if nodes is not None else ''
    h = sf.download_df_from_sql_db(
        f"SELECT dt, hr, node_num, rt_value FROM {HOURLY} WHERE dt = '{dt}'{node_clause}")
    if len(h):
        h['dt'] = pd.to_datetime(h['dt']).dt.strftime('%Y-%m-%d')
        h['rt_value'] = pd.to_numeric(h['rt_value'], errors='coerce')
    return h


def build_rt5m_features_range(d0, d1, lag=LAG, value_col=VALUE_COL, prefix=PREFIX):
    """Same result as build_rt5m_features over a contiguous range, in 2 queries not 2*n."""
    s0 = (pd.Timestamp(d0) - pd.Timedelta(days=lag)).strftime('%Y-%m-%d')
    s1 = (pd.Timestamp(d1) - pd.Timedelta(days=lag)).strftime('%Y-%m-%d')

    a = sf.download_df_from_sql_db(f"""
        SELECT dt AS src_dt, hr, node_num,
               COUNT(*)                 AS {prefix}_n,
               MAX({value_col})         AS {prefix}_max,
               MIN({value_col})         AS {prefix}_min,
               AVG({value_col})         AS {prefix}_mean,
               STDDEV_SAMP({value_col}) AS {prefix}_std,
               MAX(ABS({value_col}))    AS {prefix}_absmax
        FROM {FIVE_MIN}
        WHERE dt BETWEEN '{s0}' AND '{s1}'
        GROUP BY dt, hr, node_num""")
    h = sf.download_df_from_sql_db(f"""
        SELECT dt, hr, node_num, rt_value
        FROM {HOURLY} WHERE dt BETWEEN '{d0}' AND '{d1}'""")
    if not len(a) or not len(h):
        return pd.DataFrame()

    for c in [c for c in a.columns if c.startswith(prefix) and not c.endswith('_n')]:
        a[c] = pd.to_numeric(a[c], errors='coerce')
    a[f'{prefix}_range'] = a[f'{prefix}_max'] - a[f'{prefix}_min']
    a[f'{prefix}_spike'] = a[f'{prefix}_max'] - a[f'{prefix}_mean']
    a[f'{prefix}_drop']  = a[f'{prefix}_mean'] - a[f'{prefix}_min']

    a['dt'] = (pd.to_datetime(a['src_dt']) + pd.Timedelta(days=lag)).dt.strftime('%Y-%m-%d')
    a[f'src_dt_lag{lag}'] = pd.to_datetime(a['src_dt']).dt.strftime('%Y-%m-%d')
    a = a.drop(columns=['src_dt']).rename(
        columns={c: f'{c}_lag{lag}' for c in a.columns if c.startswith(prefix)})

    h['dt'] = pd.to_datetime(h['dt']).dt.strftime('%Y-%m-%d')
    h['rt_value'] = pd.to_numeric(h['rt_value'], errors='coerce')

    m = h.merge(a, on=['dt', 'hr', 'node_num'], how='left')
    print(f'{m["dt"].nunique()} target dts | {len(m):,} node-hours | '
          f'5-min matched {m[f"{prefix}_max_lag{lag}"].notna().mean():.1%}')
    return m

13 target dts | 487,527 node-hours | 5-min matched 100.0%


In [4]:

feat_aug = build_rt5m_features_range('2026-08-05', '2026-08-19')             # ~3s per dt, 38,616 node-hours each

# feat_aug.to_parquet('rt5m_features_aug2026.parquet', index=False)   # the window rolls off daily


NameError: name 'build_rt5m_features_range' is not defined